# Notebook 14 — Sequential & Adaptive Testing

## Background

Standard A/B testing requires you to decide the sample size in advance (power analysis) and then wait until that sample size is reached before looking at results. This is inefficient: if the treatment effect is very large, you could have stopped early; if it's very small, you're forced to a predetermined end regardless.

Sequential testing addresses this: you analyze data as it accumulates and apply a stopping rule. The test terminates as soon as the evidence is conclusive — either strong evidence for an effect, or strong evidence for no effect.

The Bayesian approach to sequential testing is philosophically natural: the posterior at any time is your current state of belief, properly updated by all data seen. The stopping rule is based on posterior properties (HDI vs ROPE, probability of practical equivalence, or expected loss) rather than p-values. Crucially, Bayesian posteriors remain valid under early stopping — no alpha-inflation correction needed.

## What You'll Learn

- Why fixed-sample tests are wasteful and sequential tests are more efficient
- Bayesian sequential testing: posterior-based stopping rules
- The ROPE-based stopping rule: stop when HDI falls inside or outside ROPE
- Probability of superiority stopping rules
- Expected loss stopping: Lindley's rule
- Adaptive sample size: group sequential designs
- Error control in Bayesian sequential tests

## Why This Matters

In continuous-deployment product environments, experiments run constantly. Waiting for a predetermined sample size is often impractical — traffic fluctuates, business priorities change, you need to ship. Sequential testing lets you stop as soon as you have enough evidence, reducing the expected number of users exposed to an inferior experience while maintaining statistical rigor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Part 1 — The Cost of Fixed-Sample Testing

Fixed-sample tests commit to a sample size `n` before observing any data. This is correct for error control, but often wasteful. If the true effect is large, you'll reach statistical confidence long before n. If the effect is zero, you'll run for n samples and then fail to reject a null that's actually true.

Let's simulate how much waste fixed-sample testing creates compared to optimal sequential stopping.

In [ ]:
def hdi(samples, prob=0.94):
    sorted_s = np.sort(samples)
    n = len(sorted_s)
    w = int(np.floor(prob * n))
    widths = sorted_s[w:] - sorted_s[:n - w]
    idx = np.argmin(widths)
    return sorted_s[idx], sorted_s[idx + w]

def rope_decision(samples, rope_lo, rope_hi, hdi_prob=0.94):
    hdi_lo, hdi_hi = hdi(samples, prob=hdi_prob)
    if hdi_lo >= rope_lo and hdi_hi <= rope_hi:
        return 'equivalent'
    elif hdi_hi <= rope_lo or hdi_lo >= rope_hi:
        return 'different'
    return 'inconclusive'

# Scenario: true effect of 2% conversion lift
true_p_ctrl = 0.10
true_p_trt  = 0.12  # 2% absolute lift
rope_lo, rope_hi = -0.005, 0.005

# Fixed-sample analysis: power analysis says n=1800 per arm for 80% power at alpha=0.05
n_fixed = 1800

# Sequential Bayesian: check every 50 observations
n_sim = 500  # simulations
check_every = 50
max_n = 2000

fixed_results  = []
bayes_seq_n    = []  # sample size when Bayesian test stopped
bayes_seq_correct = []

for sim in range(n_sim):
    ctrl_data = np.random.binomial(1, true_p_ctrl, max_n)
    trt_data  = np.random.binomial(1, true_p_trt,  max_n)
    
    # Fixed sample at n_fixed
    k_c_fixed = ctrl_data[:n_fixed].sum()
    k_t_fixed = trt_data[:n_fixed].sum()
    sc = stats.beta(1 + k_c_fixed, 1 + n_fixed - k_c_fixed).rvs(10000)
    st = stats.beta(1 + k_t_fixed, 1 + n_fixed - k_t_fixed).rvs(10000)
    fixed_p_better = np.mean(st > sc)
    fixed_results.append(fixed_p_better)
    
    # Sequential Bayesian: stop when HDI vs ROPE is conclusive
    stopped = False
    for n in range(check_every, max_n + 1, check_every):
        k_c = ctrl_data[:n].sum()
        k_t = trt_data[:n].sum()
        sc2 = stats.beta(1 + k_c, 1 + n - k_c).rvs(5000)
        st2 = stats.beta(1 + k_t, 1 + n - k_t).rvs(5000)
        diff = st2 - sc2
        decision = rope_decision(diff, rope_lo, rope_hi)
        if decision != 'inconclusive':
            bayes_seq_n.append(n)
            bayes_seq_correct.append(decision == 'different')  # true effect exists
            stopped = True
            break
    if not stopped:
        bayes_seq_n.append(max_n)
        diff_final = stats.beta(1 + trt_data.sum(), 1 + max_n - trt_data.sum()).rvs(5000) - \
                     stats.beta(1 + ctrl_data.sum(), 1 + max_n - ctrl_data.sum()).rvs(5000)
        bayes_seq_correct.append(rope_decision(diff_final, rope_lo, rope_hi) == 'different')

bayes_seq_n = np.array(bayes_seq_n)
bayes_seq_correct = np.array(bayes_seq_correct)

print(f'True effect: +{(true_p_trt - true_p_ctrl)*100:.0f}% absolute lift')
print(f'ROPE: ({rope_lo*100:.1f}%, {rope_hi*100:.1f}%)')
print()
print(f'Fixed-sample (n={n_fixed}/arm):')
print(f'  P(trt better) > 0.95: {np.mean(np.array(fixed_results) > 0.95):.2%}')
print(f'  Sample size: always {n_fixed} per arm = {2*n_fixed} total')
print()
print(f'Sequential Bayesian:')
print(f'  Correct decision rate: {bayes_seq_correct.mean():.2%}')
print(f'  Mean n per arm when stopped: {bayes_seq_n.mean():.0f}')
print(f'  Median n per arm: {np.median(bayes_seq_n):.0f}')
print(f'  Stopped at max_n: {np.mean(bayes_seq_n == max_n):.1%}')
print(f'  Savings vs fixed: {(1 - bayes_seq_n.mean() / n_fixed):.1%} fewer samples on average')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(bayes_seq_n, bins=30, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(n_fixed, color='red', lw=2, linestyle='--', label=f'Fixed n={n_fixed}')
ax.axvline(bayes_seq_n.mean(), color='steelblue', lw=2, linestyle='--',
           label=f'Sequential mean={bayes_seq_n.mean():.0f}')
ax.set_xlabel('Sample Size per Arm When Stopped')
ax.set_ylabel('Density')
ax.set_title('Sequential Testing: Sample Size Distribution\nMost experiments stop early')
ax.legend(fontsize=9)

ax = axes[1]
# Empirical CDF of stopping times
sorted_n = np.sort(bayes_seq_n)
cdf = np.arange(1, len(sorted_n) + 1) / len(sorted_n)
ax.step(sorted_n, cdf, 'steelblue', lw=2, label='Sequential (cumulative % stopped)')
ax.axvline(n_fixed, color='red', lw=2, linestyle='--', label=f'Fixed n={n_fixed}')
pct_before_fixed = np.mean(bayes_seq_n <= n_fixed)
ax.axhline(pct_before_fixed, color='steelblue', linestyle=':', alpha=0.7)
ax.set_xlabel('Sample Size per Arm')
ax.set_ylabel('Cumulative % Stopped')
ax.set_title(f'{pct_before_fixed:.0%} of sequential tests\nconclude before fixed-sample n')
ax.legend(fontsize=9)

plt.suptitle('Sequential vs Fixed-Sample Bayesian Testing\nTrue 2% lift, ROPE = +-0.5%', fontsize=12)
plt.tight_layout()
plt.show()

## Part 2 — Sequential Stopping Rules

Several stopping rules are in common use for Bayesian sequential testing:

**ROPE-based**: stop when the 94% HDI falls entirely inside or outside the ROPE
- Conclusive: HDI outside ROPE -> declare meaningful difference
- Equivalence: HDI inside ROPE -> declare practical equivalence
- Inconclusive: keep collecting

**Probability of superiority**: stop when `P(trt > ctrl) > threshold_hi` or `< threshold_lo`
- Common: stop if P(trt > ctrl) > 0.95 or < 0.05
- Simple but doesn't incorporate ROPE or effect size

**Expected loss**: Lindley's rule
- Define loss for two decisions: `d1 = deploy treatment`, `d2 = keep control`
- Stop when expected loss of the better decision is below a threshold
- More principled but requires specifying the loss function

In [ ]:
# Implement and compare three stopping rules on the same data

class BayesianSequentialTest:
    """
    Sequential Bayesian A/B test with multiple stopping rules.
    Maintains running Beta posteriors and evaluates stopping conditions.
    """
    def __init__(self, rope=(-0.005, 0.005),
                 p_superior_thresh=(0.05, 0.95),
                 expected_loss_thresh=0.001,
                 hdi_prob=0.94,
                 n_mc=10000):
        self.rope = rope
        self.p_thresh_lo, self.p_thresh_hi = p_superior_thresh
        self.el_thresh = expected_loss_thresh
        self.hdi_prob  = hdi_prob
        self.n_mc      = n_mc
        
        # Running Beta posteriors
        self.alpha_c, self.beta_c = 1, 1  # control
        self.alpha_t, self.beta_t = 1, 1  # treatment
        self.n_c, self.n_t = 0, 0
    
    def update(self, n_ctrl_new, k_ctrl_new, n_trt_new, k_trt_new):
        """Add new observations."""
        self.alpha_c += k_ctrl_new
        self.beta_c  += n_ctrl_new - k_ctrl_new
        self.alpha_t += k_trt_new
        self.beta_t  += n_trt_new - k_trt_new
        self.n_c     += n_ctrl_new
        self.n_t     += n_trt_new
    
    def evaluate(self):
        """Compute current posterior metrics."""
        sc = stats.beta(self.alpha_c, self.beta_c).rvs(self.n_mc)
        st = stats.beta(self.alpha_t, self.beta_t).rvs(self.n_mc)
        diff = st - sc
        
        # Probability of superiority
        p_trt_better = np.mean(diff > 0)
        
        # HDI vs ROPE
        hdi_lo, hdi_hi = hdi(diff, self.hdi_prob)
        hdi_inside_rope  = hdi_lo >= self.rope[0] and hdi_hi <= self.rope[1]
        hdi_outside_rope = hdi_hi <= self.rope[0] or hdi_lo >= self.rope[1]
        
        # Expected loss (simplified: expected regret of each decision)
        el_deploy   = np.mean(np.maximum(sc - st, 0))  # expected loss if deploy trt
        el_keep     = np.mean(np.maximum(st - sc, 0))  # expected loss if keep ctrl
        best_action = 'deploy' if el_deploy < el_keep else 'keep'
        min_el      = min(el_deploy, el_keep)
        
        return {
            'p_trt_better':      p_trt_better,
            'diff_mean':         diff.mean(),
            'hdi':               (hdi_lo, hdi_hi),
            'hdi_inside_rope':   hdi_inside_rope,
            'hdi_outside_rope':  hdi_outside_rope,
            'el_deploy':         el_deploy,
            'el_keep':           el_keep,
            'best_action':       best_action,
            'min_expected_loss': min_el,
            'n_total':           self.n_c + self.n_t,
        }
    
    def check_stopping(self, metrics):
        """Evaluate all three stopping rules.
        Returns dict: {rule_name: ('continue' | 'stop_trt_better' | 'stop_equivalent' | 'stop_ctrl_better')}
        """
        decisions = {}
        
        # Rule 1: Probability of superiority
        p = metrics['p_trt_better']
        if p >= self.p_thresh_hi:
            decisions['p_superior'] = 'stop_trt_better'
        elif p <= self.p_thresh_lo:
            decisions['p_superior'] = 'stop_ctrl_better'
        else:
            decisions['p_superior'] = 'continue'
        
        # Rule 2: HDI vs ROPE
        if metrics['hdi_outside_rope']:
            decisions['rope_hdi'] = ('stop_trt_better' if metrics['diff_mean'] > 0
                                     else 'stop_ctrl_better')
        elif metrics['hdi_inside_rope']:
            decisions['rope_hdi'] = 'stop_equivalent'
        else:
            decisions['rope_hdi'] = 'continue'
        
        # Rule 3: Expected loss
        if metrics['min_expected_loss'] < self.el_thresh:
            decisions['expected_loss'] = f'stop_{metrics["best_action"]}'
        else:
            decisions['expected_loss'] = 'continue'
        
        return decisions

print('BayesianSequentialTest: 3 stopping rules implemented')
print('  1. Probability of superiority (P > 0.95 or < 0.05)')
print('  2. ROPE-HDI: stop when HDI falls entirely inside or outside ROPE')
print('  3. Expected loss: stop when min(E[loss_deploy], E[loss_keep]) < threshold')

In [ ]:
# Run a single sequential test and trace the metrics over time
np.random.seed(99)

true_p_c = 0.10
true_p_t = 0.13  # 3% lift
n_max    = 2000
batch    = 50    # check every 50 obs per arm

ctrl_obs = np.random.binomial(1, true_p_c, n_max)
trt_obs  = np.random.binomial(1, true_p_t, n_max)

test = BayesianSequentialTest(
    rope=(-0.005, 0.005),
    p_superior_thresh=(0.05, 0.95),
    expected_loss_thresh=0.002,
    n_mc=8000
)

history = []
stopped = {'p_superior': None, 'rope_hdi': None, 'expected_loss': None}

for n in range(batch, n_max + 1, batch):
    # New batch
    lo = n - batch
    k_c_new = ctrl_obs[lo:n].sum()
    k_t_new = trt_obs[lo:n].sum()
    test.update(batch, k_c_new, batch, k_t_new)
    
    metrics  = test.evaluate()
    stopping = test.check_stopping(metrics)
    history.append({**metrics, **stopping, 'n_per_arm': n})
    
    for rule, decision in stopping.items():
        if stopped[rule] is None and 'stop' in decision:
            stopped[rule] = (n, decision)

print(f'True effect: +{(true_p_t - true_p_c)*100:.0f}% lift')
print('\nStopping times and decisions:')
for rule, result in stopped.items():
    if result:
        n, dec = result
        print(f'  {rule:20}: stopped at n={n:4d}/arm  ({dec})')
    else:
        print(f'  {rule:20}: did not stop before max_n={n_max}')

In [ ]:
# Visualize the sequential test evolution
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

ns = [h['n_per_arm'] for h in history]
p_better = [h['p_trt_better'] for h in history]
hdi_lo   = [h['hdi'][0] for h in history]
hdi_hi   = [h['hdi'][1] for h in history]
diff_mean = [h['diff_mean'] for h in history]
el_min   = [h['min_expected_loss'] for h in history]

# Panel 1: P(trt > ctrl)
ax = axes[0]
ax.plot(ns, p_better, 'steelblue', lw=2)
ax.axhline(0.95, color='red', linestyle='--', lw=1.5, label='Stop threshold (0.95)')
ax.axhline(0.05, color='red', linestyle='--', lw=1.5)
ax.axhline(0.5,  color='gray', linestyle=':', lw=1)
if stopped['p_superior']:
    ax.axvline(stopped['p_superior'][0], color='red', lw=2, linestyle='-.',
               label=f'Stop at n={stopped["p_superior"][0]}')
ax.set_ylabel('P(trt > ctrl)')
ax.set_title('Rule 1: Probability of Superiority')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)

# Panel 2: HDI vs ROPE
ax = axes[1]
ax.fill_between(ns, hdi_lo, hdi_hi, alpha=0.3, color='steelblue', label='94% HDI')
ax.plot(ns, diff_mean, 'steelblue', lw=2)
ax.axhspan(test.rope[0], test.rope[1], alpha=0.15, color='gray', label='ROPE')
ax.axhline(0, color='black', linestyle=':', lw=1)
if stopped['rope_hdi']:
    ax.axvline(stopped['rope_hdi'][0], color='seagreen', lw=2, linestyle='-.',
               label=f'Stop at n={stopped["rope_hdi"][0]}')
ax.set_ylabel('Difference (trt - ctrl)')
ax.set_title('Rule 2: HDI vs ROPE')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.1f}%'))

# Panel 3: Expected loss
ax = axes[2]
ax.semilogy(ns, el_min, 'purple', lw=2)
ax.axhline(test.el_thresh, color='red', linestyle='--', lw=1.5,
           label=f'Stop threshold ({test.el_thresh})')
if stopped['expected_loss']:
    ax.axvline(stopped['expected_loss'][0], color='purple', lw=2, linestyle='-.',
               label=f'Stop at n={stopped["expected_loss"][0]}')
ax.set_xlabel('n per arm')
ax.set_ylabel('Min Expected Loss')
ax.set_title('Rule 3: Expected Loss')
ax.legend(fontsize=8)

plt.suptitle('Sequential Test Evolution: Three Stopping Rules', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3 — Error Control in Sequential Tests

A natural concern: does sequential testing inflate error rates? In frequentist testing, peeking at results and stopping early (optional stopping) inflates the false positive rate. What happens with Bayesian sequential tests?

The answer depends on the stopping rule:
- **ROPE-based**: the equivalence threshold naturally controls the error rate
- **P-superior threshold**: stopping at P(trt > ctrl) > 0.95 does inflate false positives somewhat, but less than frequentist peeking
- **Expected loss**: depends on the loss function and threshold

The key insight: Bayesian posteriors are *coherent* under sequential updating. The posterior at any time is the correct Bayesian answer given the data seen so far. Whether the error rate is controlled depends on the stopping rule and its calibration.

In [ ]:
# Simulate false positive rate under H0 for each stopping rule
np.random.seed(42)

n_sims = 300  # fewer for speed
n_max_err = 1000
batch_err = 50

fp_rates = {'p_superior': 0, 'rope_hdi': 0, 'expected_loss': 0}

for sim in range(n_sims):
    # H0: equal rates
    true_p = 0.10
    ctrl = np.random.binomial(1, true_p, n_max_err)
    trt  = np.random.binomial(1, true_p, n_max_err)
    
    test_err = BayesianSequentialTest(
        rope=(-0.005, 0.005), p_superior_thresh=(0.05, 0.95),
        expected_loss_thresh=0.002, n_mc=3000
    )
    
    for rule in fp_rates:
        stopped_fp = False
        test_fp = BayesianSequentialTest(
            rope=(-0.005, 0.005), p_superior_thresh=(0.05, 0.95),
            expected_loss_thresh=0.002, n_mc=3000
        )
        for n in range(batch_err, n_max_err + 1, batch_err):
            lo = n - batch_err
            k_c = ctrl[lo:n].sum()
            k_t = trt[lo:n].sum()
            test_fp.update(batch_err, k_c, batch_err, k_t)
            metrics = test_fp.evaluate()
            stop_decisions = test_fp.check_stopping(metrics)
            if 'stop_trt_better' in stop_decisions[rule] or 'stop_ctrl_better' in stop_decisions[rule]:
                # False positive: stopped and declared a difference when none exists
                if rule == 'rope_hdi' and stop_decisions[rule] != 'stop_equivalent':
                    fp_rates[rule] += 1
                elif rule != 'rope_hdi':
                    fp_rates[rule] += 1
                stopped_fp = True
                break

print('Empirical False Positive Rate under H0 (equal rates):')
print(f'  n_simulations: {n_sims}, max_n_per_arm: {n_max_err}')
print()
for rule, fp_count in fp_rates.items():
    fp_rate = fp_count / n_sims
    print(f'  {rule:20}: {fp_rate:.3f} ({fp_count}/{n_sims})')
print()
print('Note: ROPE-HDI has the most conservative error control')
print('P-superior (0.95 threshold) acts like a one-sided alpha ~ 0.05')

## Part 4 — Adaptive Stopping in Practice

A practical sequential testing protocol for product experiments:

In [ ]:
def run_sequential_ab_test(
    ctrl_stream,
    trt_stream,
    rope=(-0.005, 0.005),
    p_threshold_win=0.95,
    p_threshold_lose=0.05,
    check_every=100,
    min_n_per_arm=200,
    max_n_per_arm=5000,
    n_mc=10000
):
    """
    Run a sequential Bayesian A/B test on streaming data.
    
    Stops when:
    1. P(trt > ctrl) > p_threshold_win -> deploy treatment
    2. P(trt > ctrl) < p_threshold_lose -> keep control
    3. HDI entirely inside ROPE -> declare equivalent
    4. n > max_n_per_arm -> inconclusive, pick by posterior mean
    
    Returns a results dict with decision and stopping metrics.
    """
    alpha_c, beta_c = 1, 1
    alpha_t, beta_t = 1, 1
    
    log = []
    n = 0
    
    for i in range(0, min(len(ctrl_stream), max_n_per_arm)):
        n += 1
        alpha_c += ctrl_stream[i]
        beta_c  += 1 - ctrl_stream[i]
        alpha_t += trt_stream[i]
        beta_t  += 1 - trt_stream[i]
        
        if n < min_n_per_arm:
            continue
        
        if n % check_every != 0:
            continue
        
        sc = stats.beta(alpha_c, beta_c).rvs(n_mc)
        st = stats.beta(alpha_t, beta_t).rvs(n_mc)
        diff = st - sc
        p_better = np.mean(diff > 0)
        hdi_lo, hdi_hi = hdi(diff, prob=0.94)
        
        log.append({
            'n': n, 'p_better': p_better,
            'hdi_lo': hdi_lo, 'hdi_hi': hdi_hi,
            'diff_mean': diff.mean()
        })
        
        # Stopping rules
        if p_better >= p_threshold_win:
            return {'decision': 'deploy_treatment', 'n_per_arm': n,
                    'p_trt_better': p_better, 'diff_mean': diff.mean(),
                    'hdi': (hdi_lo, hdi_hi), 'log': log}
        elif p_better <= p_threshold_lose:
            return {'decision': 'keep_control', 'n_per_arm': n,
                    'p_trt_better': p_better, 'diff_mean': diff.mean(),
                    'hdi': (hdi_lo, hdi_hi), 'log': log}
        elif hdi_lo >= rope[0] and hdi_hi <= rope[1]:
            return {'decision': 'equivalent', 'n_per_arm': n,
                    'p_trt_better': p_better, 'diff_mean': diff.mean(),
                    'hdi': (hdi_lo, hdi_hi), 'log': log}
    
    # Reached max_n without stopping
    sc = stats.beta(alpha_c, beta_c).rvs(n_mc)
    st = stats.beta(alpha_t, beta_t).rvs(n_mc)
    diff = st - sc
    return {'decision': 'inconclusive_at_max_n', 'n_per_arm': n,
            'p_trt_better': float(np.mean(diff > 0)),
            'diff_mean': float(diff.mean()),
            'hdi': hdi(diff, 0.94), 'log': log}

# Run on three scenarios
np.random.seed(42)
scenarios = [
    ('Large lift (+3%)',  0.10, 0.13),
    ('Small lift (+1%)',  0.10, 0.11),
    ('No effect (H0)',    0.10, 0.10),
    ('Negative (-2%)',   0.10, 0.08),
]

print(f'{"Scenario":25} {"Decision":25} {"n/arm":>8} {"P(trt>ctrl)":>12} {"Diff mean":>11}')
print('-' * 85)

for label, pc, pt in scenarios:
    ctrl_s = np.random.binomial(1, pc, 5000)
    trt_s  = np.random.binomial(1, pt, 5000)
    result = run_sequential_ab_test(ctrl_s, trt_s, n_mc=5000)
    print(f'{label:25} {result["decision"]:25} {result["n_per_arm"]:>8} '
          f'{result["p_trt_better"]:>12.3f} {result["diff_mean"]*100:>10.2f}%')

## Summary

**Sequential Bayesian Testing Advantages**:
- Stop as soon as evidence is conclusive -- reduces expected sample size
- No alpha inflation from sequential looks (posteriors update coherently)
- Natural three-way decision: better / worse / equivalent
- ROPE provides practical significance threshold

**Stopping Rule Comparison**:
| Rule | Stops on | Best for | Error control |
|------|----------|----------|---------------|
| P-superior | P(trt > ctrl) > 0.95 | Simple, clear threshold | Moderate FPR |
| ROPE-HDI | HDI outside ROPE | Practical significance | Conservative |
| Expected loss | min(E[L]) < eps | Decision-theoretic | Depends on loss |

**Recommended protocol**:
1. Set minimum n (burn-in period, e.g., 200/arm) before any stopping decisions
2. Check every 50-100 new observations
3. Use ROPE-HDI as the primary stopping rule (most interpretable)
4. Set max_n as a safety valve
5. Report full posterior when stopping, not just the stopping decision

**Next**: Notebook 15 — Capstone: `oracle` Prototype. Build a production-ready Bayesian A/B testing library integrating ROPE, sequential stopping, business loss functions, and posterior reporting.